# Pre-calculated results

In [1]:
import os, json, numpy as np

In [2]:
exp_names = ['hcp1200_t1tot2', 'dhcp_t2tot1', 'brats_t2toflair', 'synth_t1toct', 'synth_cbcttoct']
for exp_name in exp_names:
    print(exp_name)
    performance = json.load(open(f'../results/lpips/{exp_name}/attdenseunet.json', 'r'))
    mean = np.mean([score for viewname in ['coronal', 'axial', 'sagittal'] for score in performance[viewname]])
    std = np.std([score for viewname in ['coronal', 'axial', 'sagittal'] for score in performance[viewname]])
    print('\t', f'{mean:.3f}$\pm${std:.2f}')

hcp1200_t1tot2
	 0.148$\pm$0.02
dhcp_t2tot1
	 0.166$\pm$0.06
brats_t2toflair
	 0.049$\pm$0.02
synth_t1toct
	 0.223$\pm$0.05
synth_cbcttoct
	 0.250$\pm$0.06


# Generating new results with synthetic images
For this, you need to have full synthetic images paried with ground-truth images (we didn't add due to file size limitation)

In [ ]:
import os, glob, tqdm, json, numpy as np, nibabel as nib
from PIL import Image

from monai.transforms import (
    Compose,
    LoadImaged,
    CropForegroundd,
    ScaleIntensityd,
    EnsureChannelFirstd,
    SpatialPadd
)

import torch
import lpips
import torchvision.transforms.functional as TF

In [4]:
device = 'cuda'
batch_size = 128 # change this depending on gpu memory
num_workers = 8 # change this depending on # of available threads
pil_save_dir = './results/jpg'
os.makedirs(pil_save_dir, exist_ok = True)

In [3]:
gt_dir = '../../data'
syn_dir = '../../checkpoint/attdenseunet_brats_t2toflair_nearest/fold_0/test_output/'

list_fname = [f.replace(syn_dir, '').strip('/') for f in glob.glob(os.path.join(syn_dir, '**', '*.nii.gz'), recursive = True)]

trans = Compose([
    LoadImaged(keys = ['gt', 'pred']),
    EnsureChannelFirstd(keys = ['gt', 'pred']),
    ScaleIntensityd(keys = ['gt', 'pred']), # pred should already be within expected range (0 to 1)
])

lpips_fn = lpips.LPIPS(net = 'vgg').to(device)

In [2]:
performance = {viewname: [] for viewname in ['coronal', 'axial', 'sagittal']}

for fname_idx, fname in enumerate(list_fname):
    print(fname_idx, '/', len(list_fname))
    fdict = {
        'gt': os.path.join(gt_dir, fname),
        'pred': os.path.join(syn_dir, fname)
    }
    data = trans(fdict)
    gt = data['gt']
    pred = data['pred']

    score_dict_lpips = {}
    for i, viewname in enumerate(['coronal', 'axial', 'sagittal']):
        gt = gt.permute(0,2,3,1)
        pred = pred.permute(0,2,3,1)
        arr = []
        for slice_idx in range(0, gt.shape[1], batch_size):
            # make batch and make 3 channel
            batch_gt = gt.permute(1,0,2,3)[slice_idx:slice_idx+batch_size].clone().to(device).repeat(1,3,1,1)
            batch_pred = pred.permute(1,0,2,3)[slice_idx:slice_idx+batch_size].clone().to(device).repeat(1,3,1,1)
            batch_gt_lpips = batch_gt * 2 - 1
            batch_pred_lpips = batch_pred * 2 - 1
            batch_gt_fid = (batch_gt * 255).to(torch.uint8)
            batch_pred_fid = (batch_pred * 255).to(torch.uint8)
            # calculate lpips
            with torch.no_grad():
                score_batch = lpips_fn(batch_pred_lpips, batch_gt_lpips).flatten().tolist()
            arr.extend(score_batch)
        score_dict_lpips[viewname] = arr
    # record lpips
    for viewname in ['coronal', 'axial', 'sagittal']:
        performance[viewname].append(np.mean(score_dict_lpips[viewname]))

In [1]:
mean = np.mean([score for viewname in ['coronal', 'axial', 'sagittal'] for score in performance[viewname]])
std = np.std([score for viewname in ['coronal', 'axial', 'sagittal'] for score in performance[viewname]])
print(mean, std)